# CUHK-X Small Model Track — InSociEUP solution (code, weights, write-up)

**Final: private LB 0.75980, rank 60 of 326 (top 18.4%). Public 0.72636.**

Everything is open:

- **Code, run logs, every submission CSV, `inference.sh`, and a full decision record:**
  https://github.com/SanTanBan/cuhkx-small-model-track
- **Checkpoints (two 92.9 MB packages):**
  https://www.kaggle.com/datasets/santanubanerjee9/cuhkx-small-model-track-insocieup

One checkpoint holds everything that runs at inference, which is what the 100 MB
rule actually constrains: nine fused members plus the person detector.

## What the solution is

| Member | Input | Role |
|---|---|---|
| R(2+1)D-34 (IG-65M → Kinetics) | 4-channel Depth+IR **person crops**, 16 frames | the one big model, int8 |
| ST-GCN × 6 | 17-joint 3D skeleton | folds 0/2/4 of two pose normalisations |
| S3D | Thermal | |
| 1-D ResNet | 5 body-worn IMUs | |
| SSDlite320 | 4 IR frames | produces the person boxes, fp16 |

Members are pooled as a weighted **geometric mean** of their class
probabilities, with weights fitted on out-of-fold predictions only, then one
Sinkhorn step towards the training class mix.

What moved the score, in order:

1. **Person crops** from a small detector on IR, applied to the pixel-aligned depth and IR.
2. **Self-training**: test clips the ensemble labelled at p ≥ 0.6 joined training. A held-out-user fold went 0.644 → 0.712.
3. **Weight averaging ("model soup")**: several students fine-tuned from the same initialisation, averaged into *one* model — an ensemble's gain at one model's size, which is the only way to ensemble under 100 MB. Checked on held-out users before use: averages matched their prediction ensembles, and beat their parents when the parents were comparably strong.
4. **One** class-rebalancing step. A second one always lost clips.

Validation was GroupKFold on `user` throughout — never on random clips.

The repository's [decision record](https://github.com/SanTanBan/cuhkx-small-model-track/blob/main/docs/approach-and-decisions.md)
also lists what was rejected and why: int6 quantisation, distillation,
a second rebalancing step, and ordering test clips by their file timestamps
(that last one leaks labels — the hosts had already treated it as a leak, so it
was not used).

## What is actually inside a 92.9 MB package

The cell below opens the submitted checkpoint from the attached dataset and
prints its members, their input sizes and the fusion weights. No internet, no
downloads: every weight the solution uses is in this one file.

In [ ]:
import glob, os, torch

path = glob.glob('/kaggle/input/**/model_soup_S6_b1.pth', recursive=True)[0]
print(f'{os.path.basename(path)}: {os.path.getsize(path)/1e6:.1f} MB on disk')

ck = torch.load(path, map_location='cpu', weights_only=False)
meta = ck['meta']
print('fusion rule      :', meta['fusion'])
print('rebalancing steps:', meta['balance_steps'])
print('stream weights   :', {k: round(float(v), 4) for k, v in meta['weights'].items()})
print(f"person detector  : {'yes' if 'detector' in ck else 'no'}")

print(f"\n{len(ck['models'])} members")
for key, cfg in meta['members'].items():
    entries = sum(v.numel() for v in ck['models'][key].values() if hasattr(v, 'numel'))
    print(f"  {key:<36} {cfg['modality']:<12} {cfg.get('size', '-')}px "
          f"arch={str(cfg.get('arch', '-')):<14} int8={cfg.get('int8', False)} "
          f"{entries/1e6:6.1f}M stored entries")

## Reproducing a submission

```bash
git clone https://github.com/SanTanBan/cuhkx-small-model-track
cd cuhkx-small-model-track && pip install -r requirements.txt
mkdir -p checkpoints && cp .../model_soup_S6_b1.pth checkpoints/model.pth
./inference.sh /path/to/small_model_track_test final_submission.csv
```

`inference.sh` reads the raw test folders, runs the packaged detector and every
member, fuses them with the weights stored in the checkpoint and writes the CSV.
Measured at 13.0 s per clip and 1.5 GB peak RAM on two CPU threads (about 1.5 h
for all 405 clips); a T4 should take 5–15 minutes.

One caveat stated plainly: the submitted CSVs came from a three-view test-time
average computed during training, while `infer.py` averages two deterministic
views, so a re-run lands very close but is not guaranteed bit-identical.

## Licence

The video trunk is fine-tuned from IG-65M/Kinetics R(2+1)D-34 (Facebook VMZ
weights, non-commercial research licence), so the published checkpoints are for
research and verification use. No competition data is redistributed.

Questions are welcome — the full day-by-day log, including the mistakes, is in
[SUBMISSION_LOG.md](https://github.com/SanTanBan/cuhkx-small-model-track/blob/main/SUBMISSION_LOG.md).